Linear Regression 2
---

In [33]:
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [34]:
df = pd.read_csv('../Loan_default.csv')
df

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255342,8C6S86ESGC,19,37979,210682,541,109,4,14.11,12,0.85,Bachelor's,Full-time,Married,No,No,Other,No,0
255343,98R4KDHNND,32,51953,189899,511,14,2,11.55,24,0.21,High School,Part-time,Divorced,No,No,Home,No,1
255344,XQK1UUUNGP,56,84820,208294,597,70,3,5.29,60,0.50,High School,Self-employed,Married,Yes,Yes,Auto,Yes,0
255345,JAO28CPL4H,42,85109,60575,809,40,1,20.90,48,0.44,High School,Part-time,Single,Yes,Yes,Other,No,0


In [35]:
df.drop(columns='LoanID', inplace=True)

In [36]:
# converting yes and nos to 0 and 1
df[['HasMortgage', 'HasDependents', 'HasCoSigner']] = df[['HasMortgage', 'HasDependents', 'HasCoSigner']].replace({'Yes': 1, 'No': 0}).astype(int)
df

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,1,1,Other,1,0
1,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,0,0,Other,1,0
2,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,1,1,Auto,0,1
3,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,0,0,Business,0,0
4,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,0,1,Auto,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255342,19,37979,210682,541,109,4,14.11,12,0.85,Bachelor's,Full-time,Married,0,0,Other,0,0
255343,32,51953,189899,511,14,2,11.55,24,0.21,High School,Part-time,Divorced,0,0,Home,0,1
255344,56,84820,208294,597,70,3,5.29,60,0.50,High School,Self-employed,Married,1,1,Auto,1,0
255345,42,85109,60575,809,40,1,20.90,48,0.44,High School,Part-time,Single,1,1,Other,0,0


In [37]:
def get_numerics_cats(df, col_names):
    numeric = []
    categorical = []
    for col_name in col_names:
        if(df[col_name].dtype != 'str'):
            numeric.append(col_name)
        else: 
            categorical.append(col_name)
    return numeric, categorical

I think for predicting a numeric variable and not default, will look at 
- LoanAmount
- InterestRate
- CreditScore?

In [38]:
col_names = df.columns
numeric, categorical = get_numerics_cats(df, col_names)


print(numeric) # NUMERIC CURRENTLY HAS DEFAULT
print(categorical)

numeric = [n for n in numeric if (n != 'Default') and (n != 'InterestRate')]
print(numeric)

['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'HasMortgage', 'HasDependents', 'HasCoSigner', 'Default']
['Education', 'EmploymentType', 'MaritalStatus', 'LoanPurpose']
['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'LoanTerm', 'DTIRatio', 'HasMortgage', 'HasDependents', 'HasCoSigner']


In [39]:
x = df[categorical+numeric]
y = df['InterestRate']

In [40]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=123, stratify=y
)

In [41]:
preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric),
    ("cat", OneHotEncoder(
        drop="first", handle_unknown="ignore"
    ), categorical)
])

In [42]:
linreg =LinearRegression()

pipe = Pipeline(steps = [
    ("preprocess", preprocess),
    ("model", linreg)
])

pipe.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](15,)","['Education','EmploymentType','MaritalStatus',...,'HasMortgage', 'HasDependents','HasCoSigner']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,15
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. 

In [43]:
y_hat = pipe.predict(x_test)

In [46]:
model_results = x_test.copy()

model_results["observed_interest_rates"] = y_test

model_results["predicted_interest_rates"] = y_hat

model_results["residual"] = (
    model_results["observed_interest_rates"]
    -
    model_results["predicted_interest_rates"]
)

model_results.head()

,Education,EmploymentType,MaritalStatus,LoanPurpose,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,LoanTerm,DTIRatio,HasMortgage,HasDependents,HasCoSigner,observed_interest_rates,predicted_interest_rates,residual
204299,Bachelor's,Self-employed,Married,Auto,40,35497,114465,580,24,2,12,0.48,0,0,0,12.76,13.495106,-0.735106
22221,PhD,Unemployed,Single,Auto,60,110097,222650,737,82,2,60,0.68,0,1,0,16.38,13.482526,2.897474
34550,Master's,Self-employed,Married,Education,26,119118,135215,614,73,3,48,0.13,1,0,0,20.12,13.514810,6.605190
25838,PhD,Part-time,Divorced,Home,24,137054,8115,635,64,1,12,0.17,0,1,1,13.22,13.642526,-0.422526
77530,Master's,Full-time,Single,Other,32,142866,126685,651,86,1,36,0.63,1,0,1,13.36,13.408010,-0.048010


In [48]:

print("R-squared", r2_score(y_test, y_hat))
print("RMSE", np.sqrt(mean_squared_error(y_test, y_hat)))
print("MAE", mean_absolute_error(y_test, y_hat))


R-squared -9.77481069130981e-05
RMSE 6.637646713047162
MAE 5.745867327708708


In [49]:
fig = px.scatter(
    model_results,
    x="predicted_interest_rates",
    y="residual",
    opacity=0.65,
    title="Residuals Versus Predicted interest rates",
    labels={
        "predicted_interest_rates": "Predicted Rates (%)",
        "residual": "Residual: Observed − Predicted (%)",
    },
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="black"
)

fig.show()

In [52]:
feature_names = (
    pipe
    .named_steps["preprocess"]
    .get_feature_names_out()
)


In [53]:
coefficient_table = pd.DataFrame({
    "predictor": feature_names,
    "coefficient": (
        pipe
        .named_steps["model"]
        .coef_
    )
})

intercept = (
    pipe
    .named_steps["model"]
    .intercept_
)

print(f"Intercept: ${intercept:,.2f}")

coefficient_table # this is the final model

Intercept: $13.51


,predictor,coefficient
0,num__Age,-0.013112
1,num__Income,-0.015935
2,num__LoanAmount,-0.022300
3,num__CreditScore,0.007664
4,num__MonthsEmployed,0.001818
5,num__NumCreditLines,-0.003407
6,num__LoanTerm,0.008880
7,num__DTIRatio,0.003585
8,num__HasMortgage,-0.009501
9,num__HasDependents,-0.003969


In [56]:
r_squared = r2_score(
    y_test,
    y_hat
)

n = len(y_test)

p = len(feature_names)

adjusted_r_squared = (
    1
    -
    (1 - r_squared)
    *
    (n - 1)
    /
    (n - p - 1)
)

print(f"R²:          {r_squared:.3f}")
print(f"Adjusted R²: {adjusted_r_squared:.3f}")
print(f"Number of observations: {n}")
print(f"Number of predictors after encoding: {p}")

R²:          -0.000
Adjusted R²: -0.001
Number of observations: 51070
Number of predictors after encoding: 23
